# CrimeGraph AI - Model 1: Botnet Detection (Paper 8)
### Hybrid CNN-LSTM Architecture on CTU-13 & IoT-23 Network Traffic
---
This notebook trains and benchmarks the **Hybrid CNN-LSTM deep learning model** for identifying malicious botnet communications, C&C beaconing, and DDoS traffic to populate Neo4j graph nodes with flagged IP addresses.

In [ ]:
# 1. Environment Setup & Dependencies
!pip install -q torch numpy pandas scikit-learn matplotlib seaborn joblib

import os
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# 2. Dataset Generation / Loading (CTU-13 / IoT-23 Feature Schema)
FEATURE_COLUMNS = [
    "duration", "src_bytes", "dst_bytes", "src_pkts", "dst_pkts",
    "flow_rate_bytes_sec", "flow_rate_pkts_sec", "is_tcp", "is_udp",
    "is_icmp", "is_common_port", "inter_arrival_time", "syn_count",
    "fin_count", "rst_count"
]

def generate_benchmark_traffic(num_samples=10000, random_seed=42):
    np.random.seed(random_seed)
    n_benign = int(num_samples * 0.6)
    n_malicious = num_samples - n_benign
    
    dur_b = np.random.exponential(scale=3.5, size=n_benign)
    sb_b = np.random.lognormal(mean=7.0, sigma=1.2, size=n_benign)
    db_b = np.random.lognormal(mean=8.5, sigma=1.5, size=n_benign)
    sp_b = np.clip((sb_b / np.random.uniform(500, 1400, size=n_benign)).astype(int), 1, 5000)
    dp_b = np.clip((db_b / np.random.uniform(500, 1400, size=n_benign)).astype(int), 1, 10000)
    proto_b = np.random.choice(["TCP", "UDP", "ICMP"], p=[0.75, 0.23, 0.02], size=n_benign)
    cport_b = np.random.choice([1, 0], p=[0.85, 0.15], size=n_benign)
    iat_b = np.random.exponential(scale=0.5, size=n_benign)
    syn_b = np.random.poisson(lam=1.2, size=n_benign)
    fin_b = np.random.poisson(lam=1.0, size=n_benign)
    rst_b = np.random.poisson(lam=0.1, size=n_benign)
    
    # Malicious
    dur_m = np.random.exponential(scale=0.1, size=n_malicious)
    sb_m = np.random.uniform(40, 50000, size=n_malicious)
    db_m = np.random.uniform(0, 100, size=n_malicious)
    sp_m = np.random.randint(1, 500, size=n_malicious)
    dp_m = np.random.choice([0, 1], size=n_malicious)
    proto_m = np.random.choice(["TCP", "UDP"], p=[0.8, 0.2], size=n_malicious)
    cport_m = np.random.choice([1, 0], p=[0.2, 0.8], size=n_malicious)
    iat_m = np.random.exponential(scale=0.01, size=n_malicious)
    syn_m = np.random.randint(10, 500, size=n_malicious)
    fin_m = np.zeros(n_malicious, dtype=int)
    rst_m = np.random.poisson(lam=1.0, size=n_malicious)
    
    durations = np.concatenate([dur_b, dur_m])
    src_bytes = np.concatenate([sb_b, sb_m])
    dst_bytes = np.concatenate([db_b, db_m])
    src_pkts = np.concatenate([sp_b, sp_m])
    dst_pkts = np.concatenate([dp_b, dp_m])
    protocols = np.concatenate([proto_b, proto_m])
    cports = np.concatenate([cport_b, cport_m])
    iats = np.concatenate([iat_b, iat_m])
    syns = np.concatenate([syn_b, syn_m])
    fins = np.concatenate([fin_b, fin_m])
    rsts = np.concatenate([rst_b, rst_m])
    labels = np.concatenate([np.zeros(n_benign, dtype=int), np.ones(n_malicious, dtype=int)])
    
    df = pd.DataFrame({
        "duration": durations, "src_bytes": src_bytes, "dst_bytes": dst_bytes,
        "src_pkts": src_pkts, "dst_pkts": dst_pkts,
        "flow_rate_bytes_sec": (src_bytes + dst_bytes) / np.maximum(durations, 0.001),
        "flow_rate_pkts_sec": (src_pkts + dst_pkts) / np.maximum(durations, 0.001),
        "is_tcp": (protocols == "TCP").astype(int),
        "is_udp": (protocols == "UDP").astype(int),
        "is_icmp": (protocols == "ICMP").astype(int),
        "is_common_port": cports, "inter_arrival_time": iats,
        "syn_count": syns, "fin_count": fins, "rst_count": rsts,
        "label": labels
    })
    return df.sample(frac=1.0, random_state=random_seed).reset_index(drop=True)

df_traffic = generate_benchmark_traffic(10000)
print(f"Loaded {len(df_traffic)} flows. Malicious proportion: {df_traffic['label'].mean():.2%}")

In [ ]:
# 3. Sliding Sequence Windows for LSTM
class SequenceFlowDataset(Dataset):
    def __init__(self, X, y, seq_len=8):
        self.samples = []
        self.labels = []
        for i in range(0, len(X) - seq_len + 1, seq_len // 2):
            self.samples.append(X[i:i + seq_len])
            self.labels.append(1.0 if np.max(y[i:i + seq_len]) > 0 else 0.0)
        self.samples = np.array(self.samples, dtype=np.float32)
        self.labels = np.array(self.labels, dtype=np.float32)
        
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        return torch.tensor(self.samples[idx]), torch.tensor(self.labels[idx])

X = df_traffic[FEATURE_COLUMNS].values
y = df_traffic['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

train_loader = DataLoader(SequenceFlowDataset(X_train_scaled, y_train, 8), batch_size=64, shuffle=True)
test_loader = DataLoader(SequenceFlowDataset(X_test_scaled, y_test, 8), batch_size=64, shuffle=False)
print(f"Batches in train: {len(train_loader)}, test: {len(test_loader)}")

In [ ]:
# 4. Paper 8 Architecture: Hybrid CNN-LSTM
class HybridCNNLSTM(nn.Module):
    def __init__(self, num_features=15, cnn_filters=64, lstm_hidden=64):
        super().__init__()
        self.conv1 = nn.Conv1d(num_features, cnn_filters, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(cnn_filters)
        self.lstm = nn.LSTM(cnn_filters, lstm_hidden, batch_first=True, bidirectional=True)
        self.fc1 = nn.Linear(lstm_hidden * 2, 64)
        self.dropout = nn.Dropout(0.35)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x_t = x.transpose(1, 2)
        c_out = F.relu(self.bn1(self.conv1(x_t)))
        lstm_in = c_out.transpose(1, 2)
        lstm_out, _ = self.lstm(lstm_in)
        pooled, _ = torch.max(lstm_out, dim=1)
        dense = F.relu(self.fc1(pooled))
        return self.fc2(self.dropout(dense)).squeeze(-1)

model = HybridCNNLSTM().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
print(model)

In [ ]:
# 5. Training Loop & Validation
epochs = 12
for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0.0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        logits = model(bx)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(by)
        
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for bx, by in test_loader:
            bx = bx.to(device)
            p = (torch.sigmoid(model(bx)) >= 0.5).cpu().numpy()
            preds.extend(p)
            targets.extend(by.numpy())
            
    acc = accuracy_score(targets, preds)
    f1 = f1_score(targets, preds)
    print(f"Epoch {epoch:02d}/{epochs:02d} | Loss: {total_loss/len(train_loader.dataset):.4f} | Test Acc: {acc*100:.2f}% | F1: {f1:.4f}")